In [1]:
import re
from collections import defaultdict

# Load Brown noun corpus
with open("brown_nouns.txt", "r", encoding="utf-8") as f:
    words = [line.strip() for line in f if line.strip()]

print("Total entries:", len(words))
print("First 20 words:", words[:20])

Total entries: 202793
First 20 words: ['investigation', 'primary', 'election', 'evidence', 'irregularities', 'place', 'jury', 'presentments', 'charge', 'election', 'praise', 'thanks', 'manner', 'election', 'term', 'jury', 'reports', 'irregularities', 'primary', 'handful']


In [2]:
# Keep only lowercase alphabetic words
vocabulary = {
    word for word in words
    if re.fullmatch(r"[a-z]+", word)
}

print("Unique lowercase words:", len(vocabulary))

Unique lowercase words: 17053


In [3]:
def plural_forms(word):
    forms = set()

    # Rule 1: S addition
    forms.add(word + "s")

    # Rule 2: E insertion
    if word.endswith(("s", "z", "x", "ch", "sh")):
        forms.add(word + "es")

    # Rule 3: Y replacement
    if word.endswith("y") and len(word) > 1:
        forms.add(word[:-1] + "ies")
        
    return forms

In [4]:
noun_pairs = []

for singular in vocabulary:

    for plural in plural_forms(singular):

        if plural in vocabulary:
            noun_pairs.append((singular, plural))

print("Valid singular-plural pairs:", len(noun_pairs))

print("\nFirst 20 pairs:")
for pair in noun_pairs[:20]:
    print(pair)

Valid singular-plural pairs: 4353

First 20 pairs:
('stain', 'stains')
('deferment', 'deferments')
('nightclub', 'nightclubs')
('judgment', 'judgments')
('diagram', 'diagrams')
('cluck', 'clucks')
('precondition', 'preconditions')
('equation', 'equations')
('persuasion', 'persuasions')
('commissioner', 'commissioners')
('passage', 'passages')
('bumper', 'bumpers')
('dip', 'dips')
('drunk', 'drunks')
('flatness', 'flatnesses')
('bit', 'bits')
('kilometer', 'kilometers')
('infestation', 'infestations')
('preference', 'preferences')
('gelding', 'geldings')


In [5]:
class FST:
    def __init__(self):
        self.states = set()
        self.input_alphabet = set()
        self.output_alphabet = set()
        self.transitions = {}
        self.initial_state = None
        self.final_states = set()

    def add_transition(self, current, input_symbol, output_symbol, next_state):
        self.states.add(current)
        self.states.add(next_state)

        self.input_alphabet.add(input_symbol)
        self.output_alphabet.add(output_symbol)

        self.transitions[(current, input_symbol)] = (
            output_symbol,
            next_state
        )

In [6]:
fst = FST()

fst.initial_state = "q0"
fst.final_states.add("qF")

# Special states
fst.states.update({
    "q0",
    "qS",
    "qE",
    "qY",
    "qF"
})

In [7]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_word = False


root = TrieNode()

# Add all valid singular nouns
singular_words = {
    singular
    for singular, plural in noun_pairs
}

for word in singular_words:

    node = root

    for char in word:

        if char not in node.children:
            node.children[char] = TrieNode()

        node = node.children[char]

    node.is_word = True

print("Singular roots:", len(singular_words))

Singular roots: 4350


In [8]:
state_counter = 0

def new_state():
    global state_counter
    state = f"q{state_counter}"
    state_counter += 1
    return state


root.state = "q0"


def build_fst(node, current_state):

    for char, child in node.children.items():

        if not hasattr(child, "state"):
            child.state = new_state()

        next_state = child.state

        fst.add_transition(
            current_state,
            char,
            char,
            next_state
        )

        build_fst(child, next_state)


build_fst(root, "q0")

In [9]:
EPSILON = "ε"

for singular in singular_words:

    node = root

    for char in singular:
        node = node.children[char]

    state = node.state

    fst.add_transition(
        state,
        EPSILON,
        "+N+SG",
        "qF"
    )

In [10]:
for singular, plural in noun_pairs:

    if plural == singular + "s":

        node = root

        for char in singular:
            node = node.children[char]

        state = node.state

        fst.add_transition(
            state,
            "s",
            "s+N+PL",
            "qF"
        )

In [11]:
for singular, plural in noun_pairs:

    if plural == singular + "es":

        node = root

        for char in singular:
            node = node.children[char]

        state = node.state

        fst.add_transition(
            state,
            "es",
            "es+N+PL",
            "qF"
        )

In [12]:
for singular, plural in noun_pairs:

    if plural == singular[:-1] + "ies":

        node = root

        for char in singular[:-1]:
            node = node.children[char]

        state = node.state

        fst.add_transition(
            state,
            "ies",
            "ies+N+PL",
            "qF"
        )

In [13]:
print("Transition Table")
print("-" * 60)

for (state, inp), (output, next_state) in fst.transitions.items():

    print(
        f"{state:10} "
        f"{inp:5} "
        f"{output:12} "
        f"{next_state}"
    )

Transition Table
------------------------------------------------------------
q0         s     s            q0
q0         t     t            q14402
q1         a     a            q2
q2         i     i            q3
q3         n     n            q4
q3         r     r            q5
q5         c     c            q6
q6         a     a            q7
q7         s     s            q8
q8         e     e            q9
q5         w     w            q10
q10        a     a            q11
q11        y     y            q12
q2         g     g            q13
q13        e     e            q14
q2         c     c            q15
q15        k     k            q16
q15        c     c            q17
q17        a     a            q18
q18        t     t            q19
q19        o     o            q20
q2         k     k            q21
q21        e     e            q22
q2         t     t            q23
q23        e     e            q24
q24        m     m            q25
q25        e     e            q26
q26       

In [14]:
def analyze(word):

    # Singular
    if word in vocabulary:
        return f"{word}= {word}+N+SG"

    # Check valid plural pairs
    for singular, plural in noun_pairs:

        if word == plural:
            return f"{word}= {singular}+N+PL"

    return "Invalid Word"

In [15]:
test_words = [
    "fox",
    "foxes",
    "jury",
    "juries",
    "law",
    "laws",
    "try",
    "tries",
    "foxs"
]

for word in test_words:
    print(analyze(word))

fox= fox+N+SG
Invalid Word
jury= jury+N+SG
juries= juries+N+SG
law= law+N+SG
laws= laws+N+SG
try= try+N+SG
tries= tries+N+SG
Invalid Word


In [16]:
results = []

for word in words:

    result = analyze(word)

    results.append(
        f"{word} = {result}"
    )

for result in results[:100]:
    print(result)

investigation = investigation= investigation+N+SG
primary = primary= primary+N+SG
election = election= election+N+SG
evidence = evidence= evidence+N+SG
irregularities = irregularities= irregularities+N+SG
place = place= place+N+SG
jury = jury= jury+N+SG
presentments = presentments= presentments+N+SG
charge = charge= charge+N+SG
election = election= election+N+SG
praise = praise= praise+N+SG
thanks = thanks= thanks+N+SG
manner = manner= manner+N+SG
election = election= election+N+SG
term = term= term+N+SG
jury = jury= jury+N+SG
reports = reports= reports+N+SG
irregularities = irregularities= irregularities+N+SG
primary = primary= primary+N+SG
handful = handful= handful+N+SG
reports = reports= reports+N+SG
jury = jury= jury+N+SG
interest = interest= interest+N+SG
election = election= election+N+SG
number = number= number+N+SG
voters = voters= voters+N+SG
size = size= size+N+SG
city = city= city+N+SG
jury = jury= jury+N+SG
registration = registration= registration+N+SG
election = election